# Пайплайн классификации ЭЭГ с использованием MNE и неглубокого многослойного перцентрона Shallow MLP

## Исследовательская проблема и вопрос

**Проблема**: Автоматическое распознавание моторного воображения по ЭЭГ затруднено из-за сложной структуры сигналов, шумов и индивидуальных особенностей человека.

**Исследовательский вопрос**: Можно ли с помощью предобработки ЭЭГ, извлечения признаков в mu- и beta-диапазонах и модели MLP эффективно различать воображаемые движения левой и правой руки?

**Цель**: Разработать и проверить базовый алгоритм классификации моторного воображения, используя библиотеку MNE для обработки сигналов и нейросеть MLP для их классификации.

## Краткое описание проделанной работы

Этот пайплайн реализует полный цикл обработки и классификации сигналов ЭЭГ:

1. **Загрузка данных** — используем встроенный датасет MNE (PhysioNet EEG Motor Movement/Imagery Dataset) — открытый набор данных, где испытуемые представляли движения левой/правой руки.
2. **Предобработка** — фильтрация сигнала полосовым фильтром (7-30 Гц, диапазон mu/beta ритмов, связанных с моторным воображением), удаление артефактов.
3. **Эпохирование** — нарезка непрерывного сигнала на отрезки (эпохи), привязанные к событиям.
4. **Извлечение признаков** — вычисление спектральной плотности мощности (Power Spectral Density, PSD) методом Welch. На основе PSD рассчитывается логарифмическая мощность сигнала в разных частотных поддиапазонах.
5. **Классификация** — обучение Shallow MLP (неглубокий многослойный перцептрон) для разделения на классы (левая рука vs правая рука).

## 1. Установка библиотек

In [1]:
%pip install -q mne numpy scipy scikit-learn matplotlib pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Импорты и параметры эксперимента

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

from mne.datasets import eegbci
from mne.io import read_raw_edf
from mne import Epochs, events_from_annotations

from scipy.signal import welch
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    classification_report, ConfusionMatrixDisplay
)

RANDOM_STATE = 42
SUBJECT = 1
RUNS = [4, 8, 12] # runs 4, 8 и 12 соответствуют моторному воображению левой и правой руки
TARGET_SFREQ = 160.0
FILTER_LOW, FILTER_HIGH = 8.0, 30.0
EPOCH_TMIN, EPOCH_TMAX = 0.5, 3.5

DATA_DIR = Path("data_physionet")
DATA_DIR.mkdir(exist_ok=True)
mne.set_log_level("WARNING")

print("MNE version:", mne.__version__)

MNE version: 1.13.2


## 3. Загрузка сырых EEG-данных

In [3]:
edf_files = eegbci.load_data(
    SUBJECT, RUNS, path=str(DATA_DIR), update_path=True
) # скачивание EDF-файлов с PhysioNet (при первом запуске)
print("Загруженные EDF-файлы:")
for f in edf_files:
    print(f)

c:\Users\First\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Failed to download 'S001R04.edf'. Will attempt the download again 2 more times.
Failed to download 'S001R04.edf'. Will attempt the download again 1 more time.


ConnectionError: ('Connection aborted.', ConnectionResetError(10054, 'Удаленный хост принудительно разорвал существующее подключение', None, 10054, None))

## 4. Чтение EDF и объединение записей

In [ ]:
raw_list = []
for fname in edf_files:
    r = read_raw_edf(fname, preload=True, verbose="ERROR")
    eegbci.standardize(r)
    raw_list.append(r)

# MNE добавляет boundary-аннотации на местах склейки записей
raw = mne.concatenate_raws(raw_list, preload=True)
raw.set_montage("standard_1005", on_missing="ignore")

print(raw)
print("Число каналов до выбора EEG:", len(raw.ch_names))

## 5. Предобработка

- Оставляем EEG-каналы;
- Полосовой фильтр 8–30 Гц: mu и beta-активность, часто используемая в motor imagery;
- Полосовой фильтр 8–30 Гц подавляет частоты сетевой помехи 50 Гц, поскольку они лежат вне целевого диапазона;
- Пересэмплирование до 160 Гц.

In [ ]:
raw.pick("eeg")
# полосовой фильтр уже подавляет частоты вне 8–30 Гц, поэтому notch 50 Гц на данном уровне не нужен
raw.filter(
    l_freq=FILTER_LOW, h_freq=FILTER_HIGH,
    fir_design="firwin", verbose="ERROR"
)
raw.resample(TARGET_SFREQ, npad="auto", verbose="ERROR")

print("Число EEG-каналов:", len(raw.ch_names))
print("Частота дискретизации:", raw.info["sfreq"], "Hz")
print("Каналы:", raw.ch_names)

In [ ]:
# визуальная проверка нескольких каналов
raw.plot(
    duration=8,
    n_channels=min(10, len(raw.ch_names)),
    scalings="auto"
)

## 6. Извлечение событий и разбиение на эпохи

Аннотации `T1` и `T2` в выбранных runs EEGBCI обозначают два целевых условия — воображение движения левой и правой руки. Сверяем наличие обеих меток, затем строим эпохи от 0.5 до 3.5 сек после события.

In [ ]:
events, event_id = events_from_annotations(raw, verbose="ERROR")
print("Найденные event_id:", event_id)

if not {"T1", "T2"}.issubset(event_id):
    raise RuntimeError(
        f"Ожидались аннотации T1 и T2, но найдены: {event_id}. "
        "Проверьте выбранные runs и разметку EDF."
    )

event_id_motor = {
    "left_hand": event_id["T1"],
    "right_hand": event_id["T2"]
}

epochs = Epochs(
    raw,
    events,
    event_id=event_id_motor,
    tmin=EPOCH_TMIN,
    tmax=EPOCH_TMAX,
    baseline=None,
    preload=True,
    reject_by_annotation=True, # исключает эпохи, пересекающие аннотации плохих участков/границ
    picks="eeg",
    verbose="ERROR"
)

print(epochs)
print("Число эпох:", len(epochs))
print("События:", event_id_motor)

In [ ]:
# gроверим количество эпох каждого класса
class_counts = {
    name: int(np.sum(epochs.events[:, 2] == code))
    for name, code in event_id_motor.items()
}
display(pd.Series(class_counts, name="Количество эпох"))

## 7. Извлечение признаков из эпох

Для каждой эпохи и каждого канала считаем PSD методом Welch, затем интегрируем мощность в диапазонах:

- **mu:** 8–13 Гц;
- **low beta:** 13–20 Гц;
- **high beta:** 20–30 Гц.

Признак — логарифм мощности диапазона (`log10 band power`). Если EEG содержит `N` каналов, получится `N * 3` признаков на эпоху.

In [ ]:
BANDS = {
    "mu_8_13": (8, 13),
    "beta_low_13_20": (13, 20),
    "beta_high_20_30": (20, 30),
}

def extract_log_bandpower(epochs, bands=BANDS):
    data = epochs.get_data(copy=True)
    sfreq = epochs.info["sfreq"]

    feature_names = [
        f"{channel}_{band}"
        for channel in epochs.ch_names
        for band in bands
    ]

    feature_rows = []
    for epoch_data in data:
        freqs, psd = welch(
            epoch_data,
            fs=sfreq,
            nperseg=min(int(2 * sfreq), epoch_data.shape[-1]),
            axis=-1
        )

        row = []
        for channel_idx in range(epoch_data.shape[0]):
            for low, high in bands.values():
                mask = (freqs >= low) & (freqs < high)
                if mask.sum() < 2:
                    power = np.mean(psd[channel_idx, mask]) if mask.any() else 1e-20
                else:
                    power = np.trapezoid(psd[channel_idx, mask], freqs[mask])
                row.append(np.log10(power + 1e-20))
        feature_rows.append(row)

    return np.asarray(feature_rows, dtype=np.float64), feature_names


X, feature_names = extract_log_bandpower(epochs)

# 0 = левая рука, 1 = правая
y = (epochs.events[:, 2] == event_id_motor["right_hand"]).astype(int)

print("Матрица признаков X:", X.shape)
print("Вектор меток y:", y.shape)
print("Примеры признаков:", feature_names[:10])

## 8. Train/test split без утечки между runs

Случайное разбиение эпох может дать завышенную оценку (эпохи из одной записи имеют общие особенности). Поэтому используем номер исходного run как группу и стараемся оставить целые runs в train или test.

In [ ]:
boundary_samples = sorted(
    int(round(onset * raw.info["sfreq"]))
    for onset, description in zip(
        raw.annotations.onset, raw.annotations.description
    )
    if "BAD boundary" in description
)

event_samples = epochs.events[:, 0]
groups = np.searchsorted(boundary_samples, event_samples, side="right")

unique_groups, group_counts = np.unique(groups, return_counts=True)
print("Группы (runs) и число эпох:", dict(zip(unique_groups, group_counts)))

if len(unique_groups) < 2:
    raise RuntimeError(
        "Не удалось выделить несколько run-групп. "
        "Проверьте BAD boundary-аннотации и first_samp."
    )

splitter = GroupShuffleSplit(
    n_splits=1, test_size=0.33, random_state=RANDOM_STATE
)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train groups:", np.unique(groups[train_idx]))
print("Test groups:", np.unique(groups[test_idx]))

## 9. Обучение Shallow MLP

Архитектура: один скрытый слой из 32 нейронов (`hidden_layer_sizes`), ReLU, Adam.

Стандартизация признаков находится внутри sklearn `Pipeline`, поэтому `StandardScaler` обучается только на train-части.

In [ ]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(
        hidden_layer_sizes=(32,),
        activation="relu",
        solver="adam",
        alpha=1e-3,
        learning_rate_init=1e-3,
        max_iter=500,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=25,
        random_state=RANDOM_STATE
    ))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Balanced accuracy:", round(balanced_accuracy_score(y_test, y_pred), 3))
print(classification_report(
    y_test, y_pred,
    labels=[0, 1],
    target_names=["left_hand", "right_hand"],
    zero_division=0
))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    labels=[0, 1],
    display_labels=["left hand", "right hand"],
    cmap="Blues"
)
plt.title("Shallow MLP — held-out run(s)")
plt.tight_layout()
plt.show()

## 10. Результаты

В результате был разработан пайплайн обработки ЭЭГ и классификации моторного воображения с использованием MNE и MLP.

Исходные сигналы были отфильтрованы в диапазоне 8–30 Гц, разделены на эпохи, соответствующие воображению движения левой и правой руки. Для каждой эпохи методом Welch были извлечены логарифмические спектральные признаки в трёх диапазонах: mu (8–13 Гц), low beta (13–20 Гц) и high beta (20–30 Гц).

На основе полученных признаков была обучена модель MLP с одним скрытым слоем из 32 нейронов. `Accuracy` показывает долю верных ответов, а `Balanced accuracy` усредняет полноту по классам и полезна при дисбалансе.

Таким образом, удалось реализовать полный цикл обработки ЭЭГ — от загрузки исходных записей до автоматической классификации двух типов моторного воображения.

Научные выводы по одному испытуемому и трём runs делать не стоит. Для более строгой оценки нужно:
- повторить обработку для многих субъектов;
- использовать разбиение по субъектам;
- подбирать гиперпараметры только на train через внутреннюю кросс-валидацию;
- добавить контроль артефактов, референсирование и оценку устойчивости признаков.

## 11. Ссылки

- [PhysioNet EEG Motor Movement/Imagery Dataset](https://physionet.org/content/eegmmidb/1.0.0/)
- [MNE-Python documentation](https://mne.tools/stable/index.html)
- [MNE EEGBCI loader](https://mne.tools/stable/generated/mne.datasets.eegbci.load_data.html)
- [MNE Epochs API](https://mne.tools/stable/generated/mne.Epochs.html)
- [REVE: A Foundation Model for EEG (arXiv:2510.21585)](https://arxiv.org/abs/2510.21585)